# 01 · Understand the prediction problem

**NFL Player Trajectory Lab** · Real 2023 tracking data · Training-only exploration

Predict each selected player's x/y position for every requested frame after the throw.
The competition supplies the ball landing point, target receiver, and forecast horizon.
This notebook reads the reproducible artifacts produced by `nfl benchmark`; it does
not silently retrain a model. Notebook 00 is optional orientation.

[Source and run instructions](https://github.com/alvaromendizabal/nfl-player-trajectory)

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown("Run `nfl benchmark` after the data audit to create the real-data results. No synthetic result is substituted here."))

## Freeze time before fitting

Keep all frames, players and plays of a game in one partition. These dates are checked against game identifiers and locked before training. Holdout labels are excluded from the development pipeline.

In [ ]:
if READY:
    display(pd.DataFrame.from_dict(protocol["partitions"], orient="index").loc[["train", "validation", "holdout"]])
    display(Markdown("**Holdout evaluation:** " + summary["holdout_evaluation"]))

## Data coverage

Input rows describe observed motion; target rows are the positions we score. A trajectory is one player within one play, and can contain several target frames.

In [ ]:
if READY:
    display(pd.DataFrame({"Training measure": ["Games", "Plays", "Scored trajectories", "Observed rows", "Target rows"], "Count": [eda[key] for key in ["games", "plays", "trajectories", "input_rows", "target_rows"]]}))
    display(pd.DataFrame(eda["weeks"])[["file", "games", "plays", "trajectories", "input_rows", "target_rows"]])

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "eda.png")))

## Candidate features, not unverified importance claims

The implemented bank has **2,843 deterministic candidate features**. It uses observed
history, landing-relative motion, nearest teammates/opponents, receiver and passer
anchors, and forecast-time interactions. Missing-history and missing-neighbour masks
are explicit. Names, birth dates, player IDs as numerical predictors, future coordinates,
and post-play outcomes are excluded.

`nfl features` screens candidates on **training residuals only**, drops constant and
near-duplicate training features, and retains at most **64 features per challenger**.
The motion-only, landing-aware, and interaction-aware ablations share a fixed ridge
regularization and feature budget. Validation chooses a development candidate; the
48-game holdout remains reserved. A large candidate count is not evidence of accuracy.

In [ ]:
from nfl_trajectory.features import feature_catalog

catalog = feature_catalog()
display(Markdown(f"**Candidate feature count:** {len(catalog):,}"))
display(catalog.groupby("family").size().rename("Candidate features").to_frame())